## Step 1: Load 2019 flight data

Building the first delay classifier on a single representative year (2019, pre-COVID) before deciding whether to scale to more years. Loading only pre-departure-safe columns from the start — this avoids ever having leakage columns in the dataframe to accidentally use later.

In [1]:
import pandas as pd
import glob

NEEDED_COLS = ["Month", "DayOfWeek", "Reporting_Airline", "Origin", "Dest",
               "CRSDepTime", "CRSElapsedTime", "Distance", "DistanceGroup", "ArrDel15"]

files_2019 = sorted(glob.glob("../data/processed/ontime/ontime_2019_*.parquet"))
df_2019 = pd.concat(
    [pd.read_parquet(f, columns=NEEDED_COLS) for f in files_2019],
    ignore_index=True
)
df_2019 = df_2019.dropna(subset=["ArrDel15"])

print(f"Shape: {df_2019.shape}")

Shape: (7268232, 10)


## Step 2: Historical delay rate feature

Static schedule features alone are weak predictors of delay. This adds each route/carrier's own historical delay rate as a feature — using only strictly prior months' data (via `.shift(1)`) to avoid leaking future information into the prediction.

In [2]:
route_carrier_history = df_2019.groupby(["Reporting_Airline", "Origin", "Dest", "Month"])["ArrDel15"].agg(["sum", "count"]).reset_index()
route_carrier_history["cumulative_delays"] = route_carrier_history.groupby(["Reporting_Airline", "Origin", "Dest"])["sum"].cumsum().shift(1)
route_carrier_history["cumulative_flights"] = route_carrier_history.groupby(["Reporting_Airline", "Origin", "Dest"])["count"].cumsum().shift(1)
route_carrier_history["historical_delay_rate"] = (route_carrier_history["cumulative_delays"] / route_carrier_history["cumulative_flights"]).fillna(df_2019["ArrDel15"].mean())

print(route_carrier_history[["Reporting_Airline", "Origin", "Dest", "Month", "historical_delay_rate"]].head(10))

  Reporting_Airline Origin Dest  Month  historical_delay_rate
0                9E    ABE  ATL      1               0.191140
1                9E    ABE  ATL      2               0.195122
2                9E    ABE  ATL      3               0.240000
3                9E    ABE  ATL      4               0.227273
4                9E    ABE  ATL      5               0.219251
5                9E    ABE  ATL      6               0.181467
6                9E    ABE  ATL      7               0.198052
7                9E    ABE  ATL      8               0.202778
8                9E    ABE  ATL      9               0.203837
9                9E    ABE  ATL     10               0.188960


## Step 3: Merge the feature in, finalize the feature set

Joining the historical delay rate back onto the flight data, bucketing departure time into categories (rather than using raw HHMM values, which would wrongly treat 23:50 and 00:10 as maximally different), and assembling the final feature matrix.

In [3]:
df_2019 = df_2019.merge(route_carrier_history[["Reporting_Airline", "Origin", "Dest", "Month", "historical_delay_rate"]],
                         on=["Reporting_Airline", "Origin", "Dest", "Month"], how="left")
df_2019["historical_delay_rate"] = df_2019["historical_delay_rate"].fillna(df_2019["ArrDel15"].mean())

def time_bucket(hhmm):
    hour = hhmm // 100
    if hour < 6: return "late_night"
    elif hour < 12: return "morning"
    elif hour < 17: return "afternoon"
    elif hour < 21: return "evening"
    else: return "night"

df_2019["dep_time_bucket"] = df_2019["CRSDepTime"].apply(time_bucket)

FEATURES = ["Month", "DayOfWeek", "Reporting_Airline", "Origin", "Dest",
            "dep_time_bucket", "CRSElapsedTime", "Distance", "DistanceGroup",
            "historical_delay_rate"]
TARGET = "ArrDel15"

X = df_2019[FEATURES].copy()
y = df_2019[TARGET]

print(X.shape)
print(X[["historical_delay_rate"]].describe())

(7268232, 10)
       historical_delay_rate
count           7.268232e+06
mean            1.969358e-01
std             8.292716e-02
min             0.000000e+00
25%             1.418093e-01
50%             1.884984e-01
75%             2.432432e-01
max             1.000000e+00


## Step 4: Chronological train/test split and encoding

Splitting Jan-Oct (train) vs. Nov-Dec (test) chronologically rather than randomly, since delay patterns cluster in time (shared weather/holiday effects) — a random split would overstate real-world performance. Encoding categoricals: one-hot for low-cardinality fields, ordinal for high-cardinality ones (Origin/Dest) to avoid an explosion of columns.

In [4]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer

train_mask = df_2019["Month"] <= 10
X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]

print(f"Train: {len(X_train):,} rows (Jan-Oct)")
print(f"Test: {len(X_test):,} rows (Nov-Dec)")

preprocessor = ColumnTransformer([
    ("onehot", OneHotEncoder(handle_unknown="ignore"), ["Reporting_Airline", "dep_time_bucket"]),
    ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), ["Origin", "Dest"]),
], remainder="passthrough")

X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc = preprocessor.transform(X_test)

print(f"Encoded feature count: {X_train_enc.shape[1]}")

Train: 6,052,513 rows (Jan-Oct)
Test: 1,215,719 rows (Nov-Dec)
Encoded feature count: 30


## Step 5: Train and evaluate

First baseline result: adding the historical delay rate feature improved AUC from 0.5973 (schedule-only) to 0.6136.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    max_samples=0.3,
    class_weight="balanced",
    n_jobs=2,
    random_state=42,
)

print("Training...")
model.fit(X_train_enc, y_train)
print("Done.")

y_pred_proba = model.predict_proba(X_test_enc)[:, 1]
y_pred = model.predict(X_test_enc)

print(f"\nROC AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(classification_report(y_test, y_pred))

Training...
Done.

ROC AUC: 0.6136
              precision    recall  f1-score   support

         0.0       0.85      0.77      0.81   1003123
         1.0       0.25      0.37      0.30    212596

    accuracy                           0.70   1215719
   macro avg       0.55      0.57      0.55   1215719
weighted avg       0.75      0.70      0.72   1215719



## Step 6: Validate with time-series cross-validation

A single train/test split could be a lucky or unlucky slice of the year. Using `TimeSeriesSplit` (chronological folds, never testing on data the model could have trained on) to confirm the result is stable: mean AUC 0.6435 across 5 folds, with a notable pattern — later-year folds (closer to the original Nov-Dec test) perform worse, suggesting fall/winter is genuinely harder to predict with these features, not that the original split was unlucky.

In [6]:
from sklearn.model_selection import TimeSeriesSplit

# Sort by month to ensure chronological order for the splitter
df_2019_sorted_idx = df_2019.sort_values("Month").index
X_sorted = X.loc[df_2019_sorted_idx].reset_index(drop=True)
y_sorted = y.loc[df_2019_sorted_idx].reset_index(drop=True)

tscv = TimeSeriesSplit(n_splits=5)

fold_aucs = []
for fold, (train_idx, test_idx) in enumerate(tscv.split(X_sorted)):
    X_tr, X_te = X_sorted.iloc[train_idx], X_sorted.iloc[test_idx]
    y_tr, y_te = y_sorted.iloc[train_idx], y_sorted.iloc[test_idx]

    prep = ColumnTransformer([
        ("onehot", OneHotEncoder(handle_unknown="ignore"), ["Reporting_Airline", "dep_time_bucket"]),
        ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), ["Origin", "Dest"]),
    ], remainder="passthrough")

    X_tr_enc = prep.fit_transform(X_tr)
    X_te_enc = prep.transform(X_te)

    m = RandomForestClassifier(n_estimators=100, max_depth=12, max_samples=0.3,
                                 class_weight="balanced", n_jobs=2, random_state=42)
    m.fit(X_tr_enc, y_tr)
    proba = m.predict_proba(X_te_enc)[:, 1]
    auc = roc_auc_score(y_te, proba)
    fold_aucs.append(auc)
    print(f"Fold {fold+1}: AUC = {auc:.4f} (train size: {len(X_tr):,}, test size: {len(X_te):,})")

print(f"\nMean AUC: {sum(fold_aucs)/len(fold_aucs):.4f}")
print(f"Std dev: {pd.Series(fold_aucs).std():.4f}")

Fold 1: AUC = 0.6244 (train size: 1,211,372, test size: 1,211,372)
Fold 2: AUC = 0.6717 (train size: 2,422,744, test size: 1,211,372)
Fold 3: AUC = 0.6740 (train size: 3,634,116, test size: 1,211,372)
Fold 4: AUC = 0.6336 (train size: 4,845,488, test size: 1,211,372)
Fold 5: AUC = 0.6138 (train size: 6,056,860, test size: 1,211,372)

Mean AUC: 0.6435
Std dev: 0.0277
